# H5. Analysis against the pre-registration

Reads the jobs collected by H4 and applies the endpoints and decision rules frozen in H3. Every section uses only the
sessions collected so far, so it can be run after any session.

| Section | Question |
|---|---|
| H5.2 | How large are hardware fingerprints within a job, between jobs on a day, between days? (null arm) |
| H5.3 | How faithfully does the device reproduce the noiseless features? (R arm) |
| H5.4 | **Primary endpoints**: hardware gain under random assignment (R); natural-pipeline leak (C); leak with no data (N) |
| H5.5 | Secondary: noiseless, blinding windows 0 to 2 days |
| H5.6 | Replay of the null slots under other submission orders (random, chronological, by symbol, by outcome) |
| H5.7 | Measured against predicted |
| H5.8 | Optional: all four HSBC model families with their grids |

In [ ]:
N_BOOT       = 2000
ALL_FAMILIES = False      # True adds the four-family HSBC grids (slow: about an hour)
USE_LOCAL    = False      # True reads the miniature local check of H4

In [ ]:
import os, sys, json, time, math, warnings
from pathlib import Path
# Find the study folder (the one holding the hsbcrep package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd()]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "hsbcrep" / "__init__.py").exists():
                return p.resolve()
    raise RuntimeError(f"Cannot find the repository folder (the one holding hsbcrep/) from {Path.cwd()}. Set ROOT by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repository folder:", ROOT)
OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
RAW = ROOT / "data" / "raw"
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from hsbcrep import backtest as BT, design as D
BT.N_JOBS = 1
HWD = OUT / "hardware"; pre = OUT / "prereg"
plan = json.loads((pre / "plan.json").read_text())
ev = pd.read_csv(OUT / "events.csv"); F = pd.read_csv(OUT / "features.csv")
hw = np.load(OUT / "hw_events.npy")
NL = np.load(OUT / "noiseless_longer129_hw.npy"); POS = {e: k for k, e in enumerate(hw)}
NB = plan["backtest"]; FAST = {k: BT.families()[k] for k in ("LR", "XGB")}
SD_SHOT = 1 / np.sqrt(plan["shots"])
def bt_settings(n):
    return int(round(NB["train_fraction"] * n)), max(8, int(round(NB["block_fraction"] * n)))

## H5.1 Collected rows

In [ ]:
log = json.loads((HWD / ("log_local.json" if USE_LOCAL else "log.json")).read_text())
recs = []
for skey, sess in log["sessions"].items():
    if skey == "0":
        continue                                                   # dry run, not scored
    for j in sess["jobs"]:
        f = HWD / f"s{skey}_j{j['job']}_evs.npy"
        if not f.exists():
            continue
        X, Sd = np.load(f), np.load(HWD / f"s{skey}_j{j['job']}_std.npy")
        for k, (e, a) in enumerate(zip(j["events"], j["arms"])):
            recs.append({"session": int(skey), "job": j["job"], "arm": a, "event": e, "X": X[k], "sd": Sd[k]})
R = pd.DataFrame(recs)
if len(R) == 0:
    raise SystemExit("nothing collected yet")
print(R.groupby("arm").size().to_dict(), "| sessions collected:", sorted(R.session.unique()))
present = set(R.arm)

## H5.2 Fingerprints

Null-arm rows share one fixed input, so they differ only by hardware state and shot noise. Per feature, the variance
is split into within-job, between-job (same day) and between-day parts, in units of the shot-noise sd. These are the
sigma of the H3 emulator.

In [ ]:
def variance_parts(df):
    X = np.stack(df.X.values).reshape(len(df), -1)
    idx = df.assign(i=range(len(df)))
    by_job = idx.groupby(["session", "job"]).i.apply(list); by_day = idx.groupby("session").i.apply(list)
    within = np.mean([X[ix].var(0, ddof=1) for ix in by_job if len(ix) > 1], axis=0)
    mj = {k: X[ix].mean(0) for k, ix in by_job.items()}
    between_job = np.mean([np.var([mj[(s, j)] for (s2, j) in mj if s2 == s], axis=0, ddof=1)
                           for s in by_day.index if sum(1 for (s2, _) in mj if s2 == s) > 1], axis=0)
    md = np.stack([X[ix].mean(0) for ix in by_day])
    r = np.mean([len(ix) for ix in by_job]); Jd = len(by_job) / len(by_day)
    sig_job = np.sqrt(np.clip(between_job - within / r, 0, None)) / SD_SHOT
    sig_day = np.sqrt(np.clip(md.var(0, ddof=1) - between_job / Jd, 0, None)) / SD_SHOT if len(md) > 1 else np.full(X.shape[1], np.nan)
    return {"within-job sd / shot sd": float(np.median(np.sqrt(within)) / SD_SHOT),
            "sigma_job (median over features)": float(np.median(sig_job)),
            "sigma_day (median over features)": float(np.nanmedian(sig_day)),
            "features with sigma_day > 1": float(np.mean(sig_day > 1))}
fp = {}
if "N" in present:
    fp["null arm"] = variance_parts(R[R.arm == "N"])
A = R[R.arm == "anchor"]
if A.session.nunique() > 1:
    XA = np.stack(A.X.values).reshape(len(A), -1)
    fp["anchor"] = {"between-job sd / shot sd": float(np.median(XA.std(0, ddof=1)) / SD_SHOT)}
pd.DataFrame(fp).T.round(3)

## H5.3 Device against noiseless (R arm)

In [ ]:
if "R" in present:
    Dr = R[R.arm == "R"]
    H = np.stack(Dr.X.values); E = NL[[POS[e] for e in Dr.event]]
    ret = (H * E).sum((0, 1)) / (E ** 2).sum((0, 1))
    Ec, Hc = E.reshape(len(E), -1) - E.reshape(len(E), -1).mean(0), H.reshape(len(H), -1) - H.reshape(len(H), -1).mean(0)
    ok = Ec.std(0) > 1e-3
    fc = (Ec[:, ok] * Hc[:, ok]).sum(0) / np.sqrt((Ec[:, ok] ** 2).sum(0) * (Hc[:, ok] ** 2).sum(0))
    print(f"signal retained per qubit, median {np.median(ret):.3f}; feature correlation across events, median {np.median(fc):.3f}; "
          f"overall correlation {np.corrcoef(H.ravel(), E.ravel())[0, 1]:.3f}")

## H5.4 Primary endpoints (blinding 0 days)

Every arm, and the classical and noiseless baselines, go through the same rolling backtest on the same events.
Differences are paired on test events and bootstrapped over events.

In [ ]:
def arm_rows(arm):
    Da = R[R.arm == arm]
    return np.stack(Da.X.values).reshape(len(Da), -1), Da.event.values

def run(X, e, blind, fams=FAST):
    o = np.argsort(ev.t.values[e], kind="stable"); X, e = X[o], e[o]
    n_train, block = bt_settings(len(e))
    p = BT.rolling(X, ev.y.values[e], ev.t.values[e], ev.day.values[e], n_train=n_train, block=block, blind_days=blind, fams=fams)
    if len(p) == 0:                                                 # every block skipped (too few events or one class)
        return pd.DataFrame(columns=["i", "p", "y", "day", "model", "cv_auc", "event"])
    p["event"] = e[p.i.values]
    return p

def boot_diff(p1, p2, n=N_BOOT, seed=0):
    m = p1.merge(p2, on="event", suffixes=("1", "2")); y = m.y1.values
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        ix = rng.integers(0, len(m), len(m))
        if y[ix].min() != y[ix].max():
            d.append(roc_auc_score(y[ix], m.p1.values[ix]) - roc_auc_score(y[ix], m.p2.values[ix]))
    if len(m) == 0 or len(np.unique(y)) < 2 or not d:
        return np.nan, np.nan, np.nan, len(m)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return roc_auc_score(y, m.p1) - roc_auc_score(y, m.p2), lo, hi, len(m)

def boot_vs_half(p, n=N_BOOT, seed=0):
    rng = np.random.default_rng(seed); y, s = p.y.values, p.p.values; d = []
    for _ in range(n):
        ix = rng.integers(0, len(p), len(p))
        if y[ix].min() != y[ix].max():
            d.append(roc_auc_score(y[ix], s[ix]) - 0.5)
    if len(p) == 0 or len(np.unique(y)) < 2 or not d:
        return np.nan, np.nan, np.nan, len(p)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return roc_auc_score(y, s) - 0.5, lo, hi, len(p)

PRED = {a: run(*arm_rows(a), blind=0) for a in ("R", "C", "N") if a in present}
e_ref = arm_rows("R")[1] if "R" in present else hw
PRED["classical"] = run(F.values[e_ref], e_ref, 0)
rows = []
if "R" in PRED:
    b, lo, hi, n = boot_diff(PRED["R"], PRED["classical"])
    rows.append({"endpoint": "1. AUC(R) - AUC(classical)", "estimate": b, "95% CI": f"{lo:+.3f} to {hi:+.3f}", "test events": n, "declared": lo > 0.05})
if "C" in PRED and "R" in PRED:
    b, lo, hi, n = boot_diff(PRED["C"], PRED["R"])
    rows.append({"endpoint": "2. AUC(C) - AUC(R)", "estimate": b, "95% CI": f"{lo:+.3f} to {hi:+.3f}", "test events": n, "declared": lo > 0.05})
if "N" in PRED:
    b, lo, hi, n = boot_vs_half(PRED["N"])
    rows.append({"endpoint": "3. AUC(N) - 0.5", "estimate": b, "95% CI": f"{lo:+.3f} to {hi:+.3f}", "test events": n, "declared": lo > 0.05})
prim = pd.DataFrame(rows); prim.to_csv(OUT / "results_primary.csv", index=False)
print("pooled AUC:", {k: round(BT.summarize(v)["pooled_auc"], 3) for k, v in PRED.items()})
prim.round(3)

## H5.5 Secondary: noiseless and blinding windows

In [ ]:
sec = []
Xn = NL[[POS[k] for k in e_ref]].reshape(len(e_ref), -1)
for name, getter in (("classical", lambda: (F.values[e_ref], e_ref)), ("noiseless", lambda: (Xn, e_ref)),
                     *[(a, (lambda a=a: arm_rows(a))) for a in ("R", "C", "N") if a in present]):
    X, e = getter()
    for blind in NB["blinds"]:
        p = PRED[name] if (blind == 0 and name in PRED) else run(X, e, blind)
        sec.append({"input": name, "blind_days": blind, **BT.summarize(p)})
sec = pd.DataFrame(sec); sec.to_csv(OUT / "results_secondary.csv", index=False)
fig, ax = plt.subplots(figsize=(6.6, 3.8))
for name, col in (("classical", "#52514e"), ("noiseless", "#9a9893"), ("R", "#2a78d6"), ("C", "#1baf7a"), ("N", "#c0392b")):
    g = sec[sec.input == name]
    if len(g): ax.plot(g.blind_days, g.pooled_auc, marker="o", color=col, label=name)
ax.axhline(0.5, color="#c9c8c3", ls=":"); ax.set_xlabel("blinding window (days)"); ax.set_ylabel("pooled out-of-sample AUC")
ax.set_xticks(NB["blinds"]); ax.legend(frameon=False, fontsize=8); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(OUT / "fig_blinding.png", dpi=160)
sec.pivot(index="input", columns="blind_days", values="pooled_auc").round(3)

## H5.6 Replaying the null slots under other submission orders

Null-arm rows do not depend on the event, so within each session its slots can be given that market day's events in
any order, and the backtest rerun. This measures, on this device and these days, the leak each pipeline would produce
with a circuit that carries no data. "By outcome" is the class-ordered file of do Carmo et al.

In [ ]:
if "N" in present:
    Nr = R[R.arm == "N"].reset_index(drop=True).reset_index().rename(columns={"index": "row"})
    Nr["session"] = Nr.session - 1                                  # schedule index: session 1 = market day 0
    days = sorted(ev.day.unique())
    def replay_pool(rng):
        """If a session has fewer null slots than market-day events (halved null arm), draw that many of the day's
        events first, keeping the day's fill rate, so that every order reorders the same balanced sample."""
        keep = []
        for s, g in Nr.groupby("session"):
            e = hw[ev.day.values[hw] == days[s]]
            if len(g) >= len(e):
                keep.append(e); continue
            pos, neg = e[ev.y.values[e] == 1], e[ev.y.values[e] == 0]
            k1 = int(round(len(g) * len(pos) / len(e)))
            keep += [rng.choice(pos, k1, replace=False), rng.choice(neg, len(g) - k1, replace=False)]
        return np.concatenate(keep)
    sub = any(len(g) < (ev.day.values[hw] == days[s]).sum() for s, g in Nr.groupby("session"))   # null arm halved
    out = []
    for order in ("random", "chronological", "by symbol", "by outcome"):
        aucs = []
        for rep in range(5 if (order == "random" or sub) else 1):
            rng = np.random.default_rng(rep)
            pairs = D.reassign(Nr, ev, replay_pool(rng), order, rng)
            X = np.stack(Nr.X.values[[r for r, _ in pairs]]).reshape(len(pairs), -1); e = np.array([k for _, k in pairs])
            aucs.append(BT.summarize(run(X, e, 0))["pooled_auc"])
        out.append({"submission order": order, "pooled AUC": float(np.mean(aucs)), "replays": len(aucs)})
    replay = pd.DataFrame(out); replay.to_csv(OUT / "results_replay.csv", index=False)
    display(replay.round(3))

## H5.7 Measured against predicted

H3 predicted each arm's AUC for fingerprint sizes sigma = 0 to 2. H5.2 measured sigma; the nearest prediction is shown.

In [ ]:
predtab = pd.read_csv(pre / "predictions.csv").groupby(["arm", "blind_days", "sigma"]).auc.mean().reset_index()
sig = fp.get("null arm", {}).get("sigma_job (median over features)", np.nan)
near = min(predtab.sigma.unique(), key=lambda s: abs(s - sig)) if np.isfinite(sig) else None
cmp = []
for arm in ("R", "C", "N"):
    for blind in NB["blinds"]:
        m = sec[(sec.input == arm) & (sec.blind_days == blind)]
        pr = predtab[(predtab.arm == arm) & (predtab.blind_days == blind) & (predtab.sigma == near)] if near is not None else predtab.iloc[0:0]
        if len(m):
            cmp.append({"arm": arm, "blind_days": blind, "measured": float(m.pooled_auc.iloc[0]),
                        f"predicted (sigma {near})": float(pr.auc.iloc[0]) if len(pr) else np.nan})
if "N" in present:
    pr = predtab[(predtab.arm == "N replayed by outcome") & (predtab.sigma == near)]
    cmp.append({"arm": "N replayed by outcome", "blind_days": 0, "measured": float(replay.set_index("submission order").loc["by outcome", "pooled AUC"]),
                f"predicted (sigma {near})": float(pr.auc.iloc[0]) if len(pr) else np.nan})
print(f"measured sigma_job = {sig:.2f} shot-noise units; nearest prediction grid point: {near}")
pd.DataFrame(cmp).round(3)

## H5.8 Optional: all four HSBC model families

In [ ]:
if ALL_FAMILIES:
    full = BT.families(full=True)
    rows = [{"input": a, **BT.summarize(run(*arm_rows(a), blind=0, fams=full))} for a in ("R", "C", "N") if a in present]
    rows.append({"input": "classical", **BT.summarize(run(F.values[e_ref], e_ref, 0, fams=full))})
    pd.DataFrame(rows).to_csv(OUT / "results_all_families.csv", index=False)
    display(pd.DataFrame(rows).round(3))